## Problemas en Lenguaje Natural

El procesamiento de lenguaje natural (NLP, por sus siglas en inglés) aborda tareas donde el insumo principal son textos escritos por humanos. Algunos ejemplos típicos incluyen:

- **Clasificación de texto:** determinar si una reseña es positiva o negativa, identificar el tema de un documento, filtrar spam, etc.
- **Generación de texto:** producir respuestas, completar textos, generar resúmenes o traducir entre idiomas.
- **Secuencias etiquetadas:** reconocer entidades (personas, lugares, organizaciones), asignar categorías gramaticales o detectar aspectos dentro de opiniones.
- **Sistemas de búsqueda y recuperación:** encontrar documentos relevantes a partir de una consulta en lenguaje natural.
- **Modelado del lenguaje:** predecir la siguiente palabra en una secuencia o evaluar la probabilidad de una frase.

En todos estos problemas, los modelos deben transformar **texto crudo**, que es ambiguo e irregular, en representaciones numéricas que un algoritmo pueda entender y procesar. Esta transformación es el primer reto fundamental del NLP.

---

## Tokenización

La *tokenización* es el proceso que convierte un texto en una secuencia de unidades discretas llamadas **tokens**, que pueden ser:

- Palabras completas  
- Subpalabras  
- Caracteres  
- Símbolos especiales
  
Por ejemplo, el texto "Esta película fue sorprendentemente buena" puede tokenizarse como:

- **Palabras:** `["Esta", "película", "fue", "sorprendentemente", "buena"]`
- **Subpalabras:** `["Esta", "película", "fue", "sorprendente", "mente", "buena"]`
- **Caracteres:** `["T", "h", "i", "s", ...]`

Los modelos modernos utilizan **subpalabras** porque equilibran vocabularios compactos con la capacidad de representar cualquier palabra, incluso si no aparece en el conjunto de entrenamiento.

Después de tokenizar, cada token se convierte en un **índice entero** asociado a un vocabulario. Esto permite pasar del mundo del texto al mundo numérico, donde el modelo puede aprender patrones, dependencias y significados.

La tokenización es, por tanto, el puente entre el lenguaje humano y los modelos de aprendizaje profundo.


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
#Usaremos un conjunto de datos llamado IMDB con 25.000 reseñas de películas de entrenamiento y 25.000 de prueba

(X_train, y_train), (X_test, y_test) = datasets.imdb.load_data(num_words=20_000)

#Como el dataset ya viene tokenizado, vamos a usar un diccionario inverso para ver ejemplos de reseñas:

# Cargar el diccionario de palabra -> índice
word_index = datasets.imdb.get_word_index()

# Crear diccionario inverso: índice -> palabra
reverse_word_index = {value + 3: key for key, value in word_index.items()}
reverse_word_index[0] = "<PAD>"
reverse_word_index[1] = "<START>"
reverse_word_index[2] = "<UNK>"
reverse_word_index[3] = "<UNUSED>"

def decode_review(encoded_review):
    return " ".join([reverse_word_index.get(i, "?") for i in encoded_review])

print("Reseña 0 (label =", y_train[0], "):")
print(decode_review(X_train[0]))

print("\n--------------------\n")

print("Reseña 1 (label =", y_train[1], "):")
print(decode_review(X_train[1]))

2026-05-25 16:20:11.152292: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779744011.213033    3455 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779744011.229882    3455 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-25 16:20:11.348323: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Reseña 0 (label = 1 ):
<START> this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert <UNK> is an amazing actor and now the same being director <UNK> father came from the same scottish island as myself so i loved the fact there was a real connection with this film the witty remarks throughout the film were great it was just brilliant so much that i bought the film as soon as it was released for retail and would recommend it to everyone to watch and the fly fishing was amazing really cried at the end it was so sad and you know what they say if you cry at a film it must have been good and this definitely was also congratulations to the two little boy's that played the <UNK> of norman and paul they were just brilliant children are often left out of the praising list i think because the stars that play them all grown up are such a big profile for the whole film but these children are 

## Redes Neuronales Recurrentes (RNN)

Las **Redes Neuronales Recurrentes (RNN)** son modelos diseñados para procesar **secuencias**: texto, audio, series de tiempo, datos sensoriales, entre otros. A diferencia de una red densa o convolucional, una RNN incorpora una **noción de memoria**, lo que le permite usar información de pasos previos para interpretar el elemento actual de la secuencia.

En tareas secuenciales, el **orden** importa:

- Las palabras de una frase se interpretan en contexto.  
- Un sensor depende de sus mediciones previas.  
- Un sonido tiene sentido por su estructura temporal.

Una red tradicional procesa entradas **independientes**. Una RNN, en cambio, procesa la secuencia **paso a paso**, recordando lo que vio antes.

---

Una RNN recibe una secuencia:

\begin{equation}
x_1, x_2, \dots, x_T
\end{equation}

y la procesa con el mismo módulo recurrente en cada paso:

\begin{equation}
h_t = \tanh(W_x x_t + W_h h_{t-1} + b)
\end{equation}

donde:

- $ x_t $ = entrada en el tiempo $ t $  
- $ h_t $ = estado oculto (la “memoria” de la RNN)  
- $ W_x $, $ W_h $, $ b $ = pesos entrenables  
- $ h_{t-1} $ = estado oculto del paso anterior

La misma matriz de pesos se utiliza en todos los pasos de la secuencia, lo que permite que la red generalice a secuencias de distinta longitud.

---

En cada paso:

1. La RNN recibe el nuevo elemento $x_t$.  
2. Lo combina con lo que “recuerda” del pasado (el estado $h_{t-1}$).  
3. Actualiza su memoria a un nuevo estado $h_t$.  

Así, $h_t$ contiene información tanto del elemento actual como del historial previo de la secuencia.

---
Una vez calculado el estado oculto, la salida puede obtenerse con:

\begin{equation}
y_t = W_y h_t + c
\end{equation}

dependiendo de si se desea una salida para cada paso o solo al final de la secuencia.

<img src="RNN.png" width=800/>

---

Primero haremos un preprocesamiento, consistente en dejar todas las reseñas de la misma longitud. Vamos a limitar el tamaño del vocabulario a 20.000, y cada texto (reseña de IMDB) lo limitaremos a 400 palabras (truncando o rellenando): 

In [2]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Hiperparámetros básicos
max_features = 20_000   # tamaño del vocabulario
maxlen = 400            # longitud máxima de cada reseña (tokens)

# Rellenar / truncar las reseñas a longitud fija
X_train = pad_sequences(X_train, maxlen=maxlen)
X_test  = pad_sequences(X_test,  maxlen=maxlen)

X_train = np.asarray(X_train, dtype="int32")
X_test  = np.asarray(X_test,  dtype="int32")

y_train = np.asarray(y_train, dtype="float32") 
y_test  = np.asarray(y_test,  dtype="float32")

print(X_train.shape, X_test.shape)

(25000, 400) (25000, 400)


Ahora creamos una primera arquitectura recurrente:
- Una capa de *embedding* que convierte cada palabra en un vector de 128 valores numéricos.
- Una capa recurrente simple de 64 unidades
- Una capa de salida de una sola neurona con función de activación sigmoidal (clasificación binaria)

Con esta arqutectura $W_x \in \mathbb{R}^{64 \times 128}$ y $W_h \in \mathbb{R}^{64 \times 64}$

In [3]:
embedding_dim = 128

model_rnn = models.Sequential()

#Capa de entrada
model_rnn.add(layers.Input(shape=(400,)))

# Capa de embedding (convierte índices en vectores densos)
model_rnn.add(layers.Embedding(input_dim=20_000,output_dim=128,))

# Capa recurrente simple
model_rnn.add(layers.SimpleRNN(64))  # 64 neuronas

# Capa de salida (clasificación binaria para este caso)
model_rnn.add(layers.Dense(1, activation='sigmoid'))

model_rnn.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])

model_rnn.summary()

I0000 00:00:1779745242.072302    3455 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4523 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2060, pci bus id: 0000:01:00.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 400, 128)       │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 64)             │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,572,417 (9.81 MB)

 Trainable params: 2,572,417 (9.81 MB)

 Non-trainable params: 0 (0.00 B)

Entrenamiento y prueba:

In [4]:
llamada = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,              # si en 3 épocas no mejora, termina
    restore_best_weights=True
)

history_rnn = model_rnn.fit(
    X_train, y_train,
    epochs=100,
    batch_size=128,
    validation_split=0.2,
    callbacks=[llamada]
)

test_loss, test_acc = model_rnn.evaluate(X_test, y_test, verbose=0)
print(f"Accuracy en test (RNN simple): {test_acc:.3f}")

Epoch 1/100


I0000 00:00:1779745680.986690    4902 service.cc:148] XLA service 0x7092540025e0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1779745680.986938    4902 service.cc:156]   StreamExecutor device (0): NVIDIA GeForce RTX 2060, Compute Capability 7.5
2026-05-25 16:48:01.025744: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1779745681.353107    4902 cuda_dnn.cc:529] Loaded cuDNN version 92200


  5/157 ━━━━━━━━━━━━━━━━━━━━ 4s 30ms/step - accuracy: 0.5005 - loss: 0.6986

I0000 00:00:1779745682.082068    4902 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


157/157 ━━━━━━━━━━━━━━━━━━━━ 8s 39ms/step - accuracy: 0.6876 - loss: 0.5664 - val_accuracy: 0.7714 - val_loss: 0.4806
Epoch 2/100
157/157 ━━━━━━━━━━━━━━━━━━━━ 4s 29ms/step - accuracy: 0.8760 - loss: 0.2987 - val_accuracy: 0.7890 - val_loss: 0.4535
Epoch 3/100
157/157 ━━━━━━━━━━━━━━━━━━━━ 4s 29ms/step - accuracy: 0.9540 - loss: 0.1308 - val_accuracy: 0.8296 - val_loss: 0.4430
Epoch 4/100
157/157 ━━━━━━━━━━━━━━━━━━━━ 4s 28ms/step - accuracy: 0.9905 - loss: 0.0380 - val_accuracy: 0.8324 - val_loss: 0.5283
Epoch 5/100
157/157 ━━━━━━━━━━━━━━━━━━━━ 4s 29ms/step - accuracy: 0.9977 - loss: 0.0122 - val_accuracy: 0.8332 - val_loss: 0.5935
Epoch 6/100
157/157 ━━━━━━━━━━━━━━━━━━━━ 4s 29ms/step - accuracy: 0.9995 - loss: 0.0043 - val_accuracy: 0.8200 - val_loss: 0.6423
Epoch 7/100
157/157 ━━━━━━━━━━━━━━━━━━━━ 4s 29ms/step - accuracy: 0.9815 - loss: 0.0505 - val_accuracy: 0.7128 - val_loss: 0.8766
Epoch 8/100
157/157 ━━━━━━━━━━━━━━━━━━━━ 5s 31ms/step - accuracy: 0.9530 - loss: 0.1221 - val_accuracy

Ahora intentemos introduciendo dropout:

In [5]:
model_rnn = models.Sequential()

#Capa de entrada
model_rnn.add(layers.Input(shape=(400,)))

# Capa de embedding (convierte índices en vectores densos)
model_rnn.add(layers.Embedding(input_dim=max_features,output_dim=embedding_dim,))

# Capa recurrente simple
model_rnn.add(layers.SimpleRNN(64,dropout=0.1, recurrent_dropout=0.1))  # 64 neuronas y dropout

# Capa de salida (clasificación binaria para este caso)
model_rnn.add(layers.Dense(1, activation='sigmoid'))

model_rnn.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])

llamada = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,              # si en 3 épocas no mejora, termina
    restore_best_weights=True
)

history_rnn = model_rnn.fit(
    X_train, y_train,
    epochs=100,
    batch_size=128,
    validation_split=0.2,
    callbacks=[llamada]
)

test_loss, test_acc = model_rnn.evaluate(X_test, y_test, verbose=0)
print(f"Accuracy en test (RNN simple): {test_acc:.3f}")


Epoch 1/100
157/157 ━━━━━━━━━━━━━━━━━━━━ 8s 41ms/step - accuracy: 0.5385 - loss: 0.6868 - val_accuracy: 0.6214 - val_loss: 0.6559
Epoch 2/100
157/157 ━━━━━━━━━━━━━━━━━━━━ 5s 31ms/step - accuracy: 0.6684 - loss: 0.6134 - val_accuracy: 0.6256 - val_loss: 0.6231
Epoch 3/100
157/157 ━━━━━━━━━━━━━━━━━━━━ 5s 31ms/step - accuracy: 0.7356 - loss: 0.5327 - val_accuracy: 0.6512 - val_loss: 0.6046
Epoch 4/100
157/157 ━━━━━━━━━━━━━━━━━━━━ 5s 32ms/step - accuracy: 0.7882 - loss: 0.4523 - val_accuracy: 0.7256 - val_loss: 0.5718
Epoch 5/100
157/157 ━━━━━━━━━━━━━━━━━━━━ 5s 32ms/step - accuracy: 0.8519 - loss: 0.3447 - val_accuracy: 0.7532 - val_loss: 0.5358
Epoch 6/100
157/157 ━━━━━━━━━━━━━━━━━━━━ 5s 32ms/step - accuracy: 0.8878 - loss: 0.2797 - val_accuracy: 0.7906 - val_loss: 0.5260
Epoch 7/100
157/157 ━━━━━━━━━━━━━━━━━━━━ 5s 32ms/step - accuracy: 0.8729 - loss: 0.2978 - val_accuracy: 0.7556 - val_loss: 0.5955
Epoch 8/100
157/157 ━━━━━━━━━━━━━━━━━━━━ 5s 32ms/step - accuracy: 0.9204 - loss: 0.2034 - 

El val_loss mejoró pero el accuracy no. Esto indica falta de capacidad en el modelo.Debemos buscar otras alternativas.

## LSTM (Long Short-Term Memory)

Las **LSTM** son una variante avanzada de las redes recurrentes diseñada para resolver una limitación fundamental de las RNN simples: su incapacidad para aprender **dependencias de largo plazo** debido al *vanishing gradient*.  
Para lograrlo, una LSTM introduce un mecanismo de **memoria explícita** y un conjunto de **puertas** que controlan cómo fluye la información en el tiempo.

Una LSTM resuelve añade una **memoria de largo plazo** $c_t$, cuya actualización se controla mediante puertas que permiten mantener gradientes más estables.

En cada paso temporal, la celda LSTM recibe:

- la entrada actual $x_t$
- el estado oculto previo $h_{t-1}$
- el estado de memoria previo $c_{t-1}$

Produce:

- un nuevo estado oculto $h_t$
- un nuevo estado de memoria $c_t$

### 1. **Puerta de olvido**  
Decide qué parte de la memoria previa debe descartarse.

\begin{equation}
f_t = \sigma(W_f x_t + U_f h_{t-1} + b_f)
\end{equation}

### 2. **Puerta de entrada**  
Controla cuánta información nueva debe añadirse a la memoria.

\begin{equation}
i_t = \sigma(W_i x_t + U_i h_{t-1} + b_i)
\end{equation}
### 3. **Candidato de memoria**  
Contenido nuevo que podría añadirse al estado de memoria.

\begin{equation}
\tilde{c}_t = \tanh(W_c x_t + U_c h_{t-1} + b_c)
\end{equation}

### 4. **Puerta de salida**  
Determina qué parte de la memoria contribuye al estado oculto.

\begin{equation}
o_t = \sigma(W_o x_t + U_o h_{t-1} + b_o)
\end{equation}


donde $\sigma$ es la función sigmoidal.

---

## Actualizaciones de memoria y estado oculto

La LSTM combina estas señales para actualizar su memoria:

\begin{equation}
c_t = f_t \odot c_{t-1} \;+\; i_t \odot \tilde{c}_t
\end{equation}

El símbolo $ \odot $ representa el **producto elemento a elemento**  
(Hadamard product). Finalmente, el nuevo estado oculto es:

\begin{equation}
h_t = o_t \odot \tanh(c_t)
\end{equation}

---
- La **puerta de olvido** decide qué parte del pasado conservar.  
- La **puerta de entrada** controla qué nueva información se incorpora.  
- El **candidato de memoria** aporta contenido actualizado.  
- La **puerta de salida** regula qué parte de la memoria influye en la salida.

Este mecanismo permite que la memoria $c_t$ transporte información durante muchos pasos temporales sin que el gradiente se degrade.

<img src="LSTM.png" width=800/>

In [5]:
model_lstm = models.Sequential()

#Capa de entrada
model_lstm.add(layers.Input(shape=(400,)))


# Capa de embedding
model_lstm.add(layers.Embedding(
    input_dim=max_features,
    output_dim=embedding_dim
))

# Capa LSTM: más capaz que la SimpleRNN
model_lstm.add(layers.LSTM(
    64,             # 64 neuronas
    dropout=0.3,    # regularización en entradas
    recurrent_dropout=0.3  # regularización en conexiones recurrentes
))

# Capa de salida binaria
model_lstm.add(layers.Dense(1, activation='sigmoid'))

model_lstm.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model_lstm.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 400, 128)       │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,609,473 (9.95 MB)

 Trainable params: 2,609,473 (9.95 MB)

 Non-trainable params: 0 (0.00 B)

In [6]:
es = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,            # si en 2 épocas seguidas no mejora, se detiene
    restore_best_weights=True
)

history_lstm = model_lstm.fit(
    X_train, y_train,
    epochs=1000,
    batch_size=128,
    validation_split=0.2,  # 20% del train para validación
    callbacks=[es],
    verbose=1
)

test_loss, test_acc = model_lstm.evaluate(X_test, y_test, verbose=0)
print(f"Accuracy en test (LSTM): {test_acc:.3f}")

Epoch 1/1000
157/157 ━━━━━━━━━━━━━━━━━━━━ 97s 595ms/step - accuracy: 0.7459 - loss: 0.5189 - val_accuracy: 0.8388 - val_loss: 0.3961
Epoch 2/1000
157/157 ━━━━━━━━━━━━━━━━━━━━ 96s 609ms/step - accuracy: 0.8550 - loss: 0.3567 - val_accuracy: 0.8082 - val_loss: 0.4211
Epoch 3/1000
157/157 ━━━━━━━━━━━━━━━━━━━━ 96s 610ms/step - accuracy: 0.8883 - loss: 0.2842 - val_accuracy: 0.8422 - val_loss: 0.3830
Epoch 4/1000
157/157 ━━━━━━━━━━━━━━━━━━━━ 96s 610ms/step - accuracy: 0.9096 - loss: 0.2419 - val_accuracy: 0.8434 - val_loss: 0.3977
Epoch 5/1000
157/157 ━━━━━━━━━━━━━━━━━━━━ 95s 608ms/step - accuracy: 0.9280 - loss: 0.2004 - val_accuracy: 0.7910 - val_loss: 0.5865
Epoch 6/1000
157/157 ━━━━━━━━━━━━━━━━━━━━ 97s 616ms/step - accuracy: 0.9337 - loss: 0.1794 - val_accuracy: 0.8046 - val_loss: 0.4724
Epoch 7/1000
157/157 ━━━━━━━━━━━━━━━━━━━━ 95s 608ms/step - accuracy: 0.9454 - loss: 0.1525 - val_accuracy: 0.8168 - val_loss: 0.5073
Epoch 8/1000
157/157 ━━━━━━━━━━━━━━━━━━━━ 96s 615ms/step - accuracy: 

## Transformers

Los **Transformers** son una arquitectura diseñada para procesar secuencias sin recurrencia y sin convoluciones.  
A diferencia de las RNN y LSTM, que procesan la secuencia paso a paso, los Transformers pueden analizar **toda la secuencia en paralelo**, lo cual los hace más rápidos, escalables y capaces de capturar dependencias largas.

Fueron introducidos en 2017 en el artículo *Attention is All You Need*, y desde entonces se han convertido en la base de los modelos modernos de NLP (BERT, GPT, T5, LLaMA, etc.).

---

Las RNN y LSTM presentan dos limitaciones importantes:

- Procesan la secuencia de manera **secuencial**: no se pueden paralelizar.
- Tienen dificultades para aprender **dependencias muy largas**, incluso con mecanismos de memoria.

Los Transformers resuelven ambos problemas utilizando **self-attention**, que permite que cada token se relacione con todos los demás directamente.

---

### Self-Attention

El mecanismo central del Transformer es la **auto-atención** (*self-attention*).  
Para cada token de la entrada, el modelo decide **qué otros tokens son relevantes**, asignando pesos de atención.

Supongamos que tenemos secuencias de $n$ tokens que se transforman a un embebimiento de $D$ dimensiones distribuidas en cabezas de $d$ dimensiones cada una ($d \times n_{cabezas} = D$). 

Cada token se transforma en tres vectores **q**, **k** y **v** que serán las filas de las matrices:

- **Q**: Query (consulta) $\in  \mathbb{R}^{n\times d}$ 
- **K**: Key (clave) $\in  \mathbb{R}^{n\times d}$  
- **V**: Value (valor) $\in  \mathbb{R}^{n\times d}$ 

La atención se calcula como:

\begin{equation}
\text{Attention}(Q, K, V) =
\text{softmax}\left( \frac{Q K^T}{\sqrt{d}} \right) V
\end{equation}

- $QK^T$ mide la similitud entre tokens.  
- `softmax` convierte esas similitudes en pesos de atención.  
- Los pesos se aplican a los vectores $V$, produciendo una representación contextualizada.

---

### Multi-Head Attention

En vez de tener una sola atención, los Transformers utilizan varias **cabezas de atención** en paralelo:

\begin{equation}
\text{MultiHead}(Q,K,V)=
\text{Concat}(\text{head}_1,\dots,\text{head}_h)W^O
\end{equation}

Cada cabeza aprende a enfocarse en distintos tipos de relaciones:

- dependencias gramaticales  
- relaciones semánticas  
- negaciones  
- co-referencias  

Esto hace que el modelo capture patrones complejos de manera muy eficiente.

---

### Positional Encoding

Como los Transformers no procesan tokens en orden secuencial, necesitan una forma de incorporar el **orden** de la secuencia.

Esto se logra sumando un **positional encoding** al embedding:

\begin{equation}
X_{\text{final}} = \text{Embedding}(x) + \text{PositionalEncoding}
\end{equation}

El encoding puede ser senoidal (BERT original) o aprendido (como en muchos modelos actuales).

---

Cada bloque encoder consiste en:

1. **Multi-Head Self-Attention**
2. **Suma residual**  
3. **Layer Normalization**
4. **Feed-Forward Network** (pequeña red densa aplicada a cada token)
5. **Suma residual**
6. **Layer Normalization**

### Modelos generativos: Encoder + Decoder


<img src="transformers.png" width=800/>

In [7]:
#Capa de embedding con codificación posicional

class PositionalEmbedding(layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.token_emb = layers.Embedding(vocab_size, embed_dim)
        self.pos_emb = layers.Embedding(maxlen, embed_dim)

    def call(self, x):
        seq_len = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=seq_len, delta=1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x + positions

#Bloque Transformer encoder (self-attention + FFN)

class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.2):
        super().__init__()
        self.att = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim,
            dropout=rate
        )
        self.ffn = models.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dropout(rate),
            layers.Dense(embed_dim),
        ])
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout = layers.Dropout(rate)

    def call(self, x, training=None):
        # PreNorm → Attention → residual
        attn_input = self.layernorm1(x)
        attn_output = self.att(attn_input, attn_input, training=training)
        x = x + self.dropout(attn_output, training=training)

        # PreNorm → FFN → residual
        ffn_input = self.layernorm2(x)
        ffn_output = self.ffn(ffn_input, training=training)
        return x + self.dropout(ffn_output, training=training)

In [8]:
embed_dim = 128
num_heads = 4
ff_dim = 256
num_layers = 2

inputs = layers.Input(shape=(maxlen,))
x = PositionalEmbedding(maxlen, max_features, embed_dim)(inputs)

for _ in range(num_layers):
    x = TransformerBlock(embed_dim, num_heads, ff_dim, rate=0.2)(x)

x = layers.LayerNormalization(epsilon=1e-6)(x)
x = layers.GlobalAveragePooling1D()(x)

x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model_trans_medium = models.Model(inputs, outputs)

model_trans_medium.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model_trans_medium.summary()



Model: "functional_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 400)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ positional_embedding            │ (None, 400, 128)       │     2,611,200 │
│ (PositionalEmbedding)           │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block               │ (None, 400, 128)       │       330,240 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_1             │ (None, 400, 128)       │       330,240 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer_normalization_4           │ (None, 400, 128)       │           256 │
│ (LayerNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,288,577 (12.54 MB)

 Trainable params: 3,288,577 (12.54 MB)

 Non-trainable params: 0 (0.00 B)

In [9]:
es_trans_med = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history_trans_med = model_trans_medium.fit(
    X_train, y_train,
    epochs=20,
    batch_size=64,     # OJO: 64, no 128
    validation_split=0.2,
    callbacks=[es_trans_med],
    verbose=1
)

Epoch 1/20


2026-05-25 17:15:41.175525: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'input_add_reduce_fusion_4', 32 bytes spill stores, 32 bytes spill loads
ptxas warning : Registers are spilled to local memory in function 'input_multiply_reduce_fusion', 8 bytes spill stores, 8 bytes spill loads



313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step - accuracy: 0.5448 - loss: 0.6961

2026-05-25 17:16:17.737234: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'input_add_reduce_fusion_4', 32 bytes spill stores, 32 bytes spill loads
ptxas warning : Registers are spilled to local memory in function 'input_multiply_reduce_fusion', 8 bytes spill stores, 8 bytes spill loads



313/313 ━━━━━━━━━━━━━━━━━━━━ 53s 131ms/step - accuracy: 0.6478 - loss: 0.5898 - val_accuracy: 0.8654 - val_loss: 0.3477
Epoch 2/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 33s 105ms/step - accuracy: 0.9020 - loss: 0.2542 - val_accuracy: 0.8918 - val_loss: 0.2971
Epoch 3/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 33s 105ms/step - accuracy: 0.9435 - loss: 0.1634 - val_accuracy: 0.8854 - val_loss: 0.3532
Epoch 4/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 33s 106ms/step - accuracy: 0.9712 - loss: 0.0918 - val_accuracy: 0.8834 - val_loss: 0.4579
Epoch 5/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 33s 106ms/step - accuracy: 0.9851 - loss: 0.0526 - val_accuracy: 0.8816 - val_loss: 0.6201
Epoch 6/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 34s 107ms/step - accuracy: 0.9915 - loss: 0.0337 - val_accuracy: 0.8774 - val_loss: 0.6493
Epoch 7/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 34s 108ms/step - accuracy: 0.9933 - loss: 0.0241 - val_accuracy: 0.8702 - val_loss: 0.8077


In [12]:
test_loss, test_acc = model_trans_medium.evaluate(X_test, y_test, verbose=0)
print(f"Accuracy en test (Transformer): {test_acc:.3f}")

Accuracy en test (Transformer): 0.880
